In [33]:
import geopandas as gpd
import pandas as pd
import numpy as np

In [20]:
gdf = gpd.read_file("Data/kd_z_nevarnost_enriched_verified.geojson")
gdf = gdf.drop(columns=['geometry'])

In [34]:
embed_cols = gdf[['IME', 'SINONIMI', 'OPIS', 'ZVRST', 'TIP', 'GESLA', 'DATACIJA', 'LOKACIJAOPIS', 'prevladujoci_material', 'UE_UIME', 'OBCINA']]
meta_data_cols = gdf[['EID', 'OBCINA', 'STATUS', 'SPOMENIK', 'UE_UIME', 'prevladujoci_material', 'pozar_ocena_popravljena', 'poplave_ocena_popravljena',
       'potres_ocena_popravljena', 'plazovi_ocena_popravljena']]

In [35]:
def row_to_text(row):
    parts = []
    for col, val in row.items():
        #codex checks za nan value ipd.
        if val is None:
            continue
        if isinstance(val, (list, tuple, np.ndarray, pd.Series)):
            if len(val) == 0 or pd.isna(val).all():
                continue
            text = " ".join(map(str, val)).strip()
            if not text:
                continue
        else:
            if pd.isna(val) or not str(val).strip():
                continue
        
        if col == 'prevladujoci_material':
            parts.append(f"material: {val}")
        elif col == 'UE_UIME':
            parts.append(f"okraj: {val}")
        else:
            parts.append(f"{col.lower()}: {val}")

    return " | ".join(parts)

In [38]:
gdf['embed_text'] = embed_cols.apply(row_to_text, axis=1)

In [42]:
gdf['embed_text'].str.len().describe()

count    31086.000000
mean       551.748408
std         92.397001
min        273.000000
25%        487.000000
50%        548.500000
75%        611.000000
max       1075.000000
Name: embed_text, dtype: float64

In [ ]:
zapisi = []

for _, row in gdf.iterrows():
    zapisi.append({
        "eid" : row['EID'],
        "text" : row['embed_text'],
        "meta_data" : {col: row[col] for col in meta_data_cols}
    })

In [ ]:
import os
from openai import AzureOpenAI
import chromadb

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_version="2024-02-01",
)

EMBED_MODEL='MDML-TextEmbedding-003'

In [ ]:
chroma_client = chromadb.PersistentClient(path="Data/chroma_db")
collection = chroma_client.get_or_create_collection("kulturna dediscina")

BATCH_SIZE = 100
for i in range(0, len(zapisi), BATCH_SIZE):
    batch = zapisi[i: i+BATCH_SIZE]
    texts = [r['text'] for r in batch]

    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )

    collection.add(
        ids=[str(r['eid']) for r in batch],
        documents=[r['text'] for r in batch],
        metadatas=[r['meta_data'] for r in batch],
        embeddings=[item.embedding for item in response.data]
    )

NameError: name 'client' is not defined